# Exploration: Determinants with panchi

**Name:** Your name here

In this exploration we'll use [panchi](https://github.com/Gustavo-Galvao-e-Silva/panchi), a Python linear algebra library built for *learning*, to explore the ideas from Sections 3.1–3.3: computing determinants, how row operations change them, and what they mean geometrically.

**No coding experience needed.** Most problems only ask you to replace a `___` blank with a number or a short expression.

**How to complete this exploration**

1. Open the notebook in Google Colab and choose **File → Save a copy in Drive**, then work in that copy. (Skip this if the notebook was handed to you as your own copy, e.g. through Google Classroom.)
2. **Run the setup cell below once before beginning.** Then run the rest of the cells **from top to bottom**. Click a cell and press **Shift + Enter** to run it.
3. Wherever you see `___`, replace it with your answer and run the cell again.
4. Answer the written questions by double-clicking the *Your answer* cells and typing.
5. When you're done, run the **Export to PDF** cell at the very end and submit the PDF.

### Python in 60 seconds

- `x = 5` **stores** the value `5` under the name `x`. From then on, `x` means `5`.
- `print(x)` shows a value below the cell.
- Lines starting with `#` are **comments**. Python ignores them; they're notes for you.
- Python counts from **0**, not 1. The first row of a matrix is row `0`, the second is row `1`, and so on.
- If a cell says *"This cell still has a ___ blank"*, there's still something to fill in!
- Errors are normal and nothing breaks: read the last line of the message, fix the cell, and run it again.

## ⚠️ RUN THIS SETUP CELL ONCE BEFORE BEGINNING THE EXPLORATION ⚠️

It installs panchi and the PDF tools, then asks for permission to access your Google Drive, so that the export cell at the very end can find your saved notebook. It takes a minute or two, and nothing else in the notebook will work until it has finished.

- Only run it **once**. You only need to run it again if Colab disconnects or you come back in a new session.
- It finds your saved copy of this notebook automatically. If you **renamed** it, put part of the new name in `NOTEBOOK_NAME` before running.

In [ ]:
# ===== RUN THIS SETUP CELL ONCE BEFORE BEGINNING THE EXPLORATION =====

import glob
import io
import os
import tokenize

# 1. Install panchi, plus pandoc and LaTeX for the PDF export at the end
%pip install -q "panchi>=2.0"
!apt-get -qq update > /dev/null
!apt-get -qq install -y texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc > /dev/null

# 2. Connect your Google Drive (the PDF is made from the copy saved there)
from google.colab import drive

drive.mount("/content/drive")

# 3. Find your saved copy of this notebook in Drive. It's found even if it's called
#    "Copy of ..." or has your name in front. Only change this if you renamed it!
NOTEBOOK_NAME = "exploration_determinants"
OUTPUT_FOLDER = "/content/drive/MyDrive"

found = []
for folder in ["Colab Notebooks", "Classroom"]:
    pattern = f"{OUTPUT_FOLDER}/{folder}/**/*{NOTEBOOK_NAME}*.ipynb"
    found += glob.glob(pattern, recursive=True)

if found:
    NOTEBOOK = max(found, key=os.path.getmtime)  # the most recently saved one
    print("Setup complete! Your notebook is:", NOTEBOOK)
else:
    NOTEBOOK = None
    print("Setup finished, but no saved copy of", NOTEBOOK_NAME, "was found in your Drive.")
    print("Choose File > Save a copy in Drive, then run this cell again.")


# 4. Friendly reminder instead of a confusing error when a cell still has a ___ blank
#    (IPython normally uses ___ as a name for an old output, so blanks wouldn't fail)
def _check_blanks(lines):
    try:
        tokens = tokenize.generate_tokens(io.StringIO("".join(lines)).readline)
        has_blank = any(t.type == tokenize.NAME and t.string == "___" for t in tokens)
    except Exception:
        return lines
    if has_blank:
        return ['print("This cell still has a ___ blank. '
                'Replace every ___ with your answer, then run it again.")\n']
    return lines


get_ipython().input_transformers_cleanup.append(_check_blanks)

In [ ]:
import panchi as pan
from panchi.algorithms import RowSwap, RowScale, RowAdd
from panchi.visualizations import Animator2D

animator = Animator2D(figsize=(5, 5))

## Part A · Matrices in panchi

A matrix is created from a list of its rows. Here is the matrix from **Example 3.3**. If a panchi object is the last line of a cell, it's displayed as a nicely typeset matrix:

In [ ]:
A = pan.Matrix([
    [1, -5,  2],
    [2,  4, -1],
    [0,  3,  0],
])
A

We read single entries with **square brackets** and a `(row, column)` pair. Because Python counts from 0, the math entry $a_{12}$ (row 1, column 2) is `A[0, 1]` in code:

In [ ]:
print("number of rows:", A.rows)
print("first row:     ", A[0])
print("a_12 =", A[0, 1])

### Problem 1. Reading entries

Fill in the blanks so that `a23` holds the entry $a_{23}$ (row 2, column 3) and `a31` holds $a_{31}$ (row 3, column 1). Then compare the printed values with the matrix above.

In [ ]:
a23 = A[___, ___]
a31 = A[___, ___]

print("a_23 =", a23)
print("a_31 =", a31)

## Part B · Your own 2×2 determinant

From **Example 3.2**, the determinant of a $2 \times 2$ matrix is

$$\det\begin{bmatrix} a & b \\ c & d \end{bmatrix} = ad - bc.$$

Below is a **function**: a reusable recipe that takes a matrix `M` and *returns* its determinant. `def` starts the recipe, and the indented lines are its steps. Two entries are already pulled out for you.

### Problem 2. Complete `det2`

Fill in the blanks to get `c` and `d`, then write the formula on the `return` line using `a`, `b`, `c`, `d`. (In Python, `*` means multiply.)

In [ ]:
def det2(M):
    a = M[0, 0]
    b = M[0, 1]
    c = ___
    d = ___
    return ___

Now run the checker below. It compares your `det2` with panchi's built-in `pan.determinant` on a few matrices.

In [ ]:
tests = [
    pan.Matrix([[1, 2], [3, 4]]),
    pan.Matrix([[3, -1], [5, 2]]),
    pan.Matrix([[2, 4], [1, 2]]),
]

for M in tests:
    try:
        mine = det2(M)
    except NameError:
        print("There's still a ___ to fill in inside det2!")
        break
    expected = pan.determinant(M)
    status = "PASS" if mine == expected else "NOT YET"
    print(f"{M.to_list()}:  det2 = {mine},  panchi = {expected}  ->  {status}")

## Part C · Row operations change the determinant

panchi has the three elementary row operations built in: `RowSwap`, `RowScale`, and `RowAdd`. Calling `.apply(B)` gives a **new** matrix with the operation performed (rows are numbered from 0 here too).

Let's use the matrix from **Example 3.11** and swap its first two rows:

In [ ]:
B = pan.Matrix([
    [ 1, -4,  2],
    [-2,  8, -9],
    [-1,  7,  3],
])

swapped = RowSwap(0, 1).apply(B)
swapped

In [ ]:
print("det(B)       =", pan.determinant(B))
print("det(swapped) =", pan.determinant(swapped))

Notice how Examples 3.3 and 3.4 in the notes are the same matrix with rows 1 and 3 swapped, and their determinants are 15 and −15.

### Problem 3. Scaling and adding rows

Pick your own numbers! Replace the blanks with any **non-zero** numbers you like (try something like `3` or `-2`), then run the cell.

In [ ]:
# Multiply row 2 of B by a number k
k = ___
scaled = RowScale(row=2, scalar=k).apply(B)

# Add (some number) times row 0 to row 1
added = RowAdd(target=1, source=0, scalar=___).apply(B)

print("k =", k)
print("det(B)      =", pan.determinant(B))
print("det(scaled) =", pan.determinant(scaled))
print("det(added)  =", pan.determinant(added))

### Question 4. Fill in the blanks

Using what you saw above (and Theorem 3.10), complete each sentence. Double-click this cell to edit it.

- Swapping two rows multiplies the determinant by **\_\_\_\_\_**.
- Multiplying one row by $k$ multiplies the determinant by **\_\_\_\_\_**.
- Adding a multiple of one row to another multiplies the determinant by **\_\_\_\_\_**.

## Part D · The determinant as area

From Section 3.3: the parallelogram spanned by the columns of a $2 \times 2$ matrix $M$ has area $|\det(M)|$.

### Problem 5. The parallelogram from Example 3.19

The parallelogram has corners $P_1 = (-2, 3)$, $P_2 = (1, 7)$, $P_3 = (6, -3)$ and $P_4 = (9, 1)$. Its two edges starting at $P_1$ are the vectors $P_2 - P_1$ and $P_3 - P_1$.

Complete the **TODO** lines. Hints:

- Vectors can be subtracted just like numbers: `P2 - P1`.
- `pan.vector_column_matrix([v1, v2])` builds the matrix whose columns are `v1` and `v2`.
- `abs(x)` is the absolute value of `x`.

In [ ]:
P1 = pan.Vector([-2, 3])
P2 = pan.Vector([1, 7])
P3 = pan.Vector([6, -3])
P4 = pan.Vector([9, 1])

# TODO: the two edge vectors starting at P1
v1 = ___
v2 = ___

# TODO: put v1 and v2 as the columns of a matrix M, then use your det2
M = ___
area = ___

print("v1 =", v1, "  v2 =", v2)
print("area =", area)

Let's see it. The parallelogram's sides are `v1` and `v2`, and its far corner is `v1 + v2` (which is exactly $P_4 - P_1$).

In [ ]:
animator.plot_vectors(
    [v1, v2, v1 + v2],
    labels=["v1", "v2", "v1 + v2"],
)

Now let's transform the parallelogram with the matrix $T = \begin{bmatrix} 1 & 1 \\ 0 & 2 \end{bmatrix}$. Every vector $x$ is sent to $Tx$, so the new parallelogram is spanned by $Tv_1$ and $Tv_2$.

In [ ]:
T = pan.Matrix([[1, 1], [0, 2]])

new_M = pan.vector_column_matrix([T @ v1, T @ v2])

print("det(T)   =", det2(T))
print("old area =", abs(det2(M)))
print("new area =", abs(det2(new_M)))

animator.plot_vectors(
    [T @ v1, T @ v2, T @ (v1 + v2)],
    labels=["T v1", "T v2", "T (v1 + v2)"],
)

**Question.** How is the new area related to the old area and to $\det(T)$? Answer in one sentence (see Theorem 3.21).

*Your answer:* 

## Challenge (optional): cofactor expansion for 3×3

The definition in Section 3.1 expands along the first row:

$$\det(A) = a_{11}\det(A_{11}) - a_{12}\det(A_{12}) + a_{13}\det(A_{13}),$$

where $A_{1j}$ is $A$ with row 1 and column $j$ deleted. The helper `minor(M, i, j)` below builds that smaller matrix for you (remember: counting from 0). You don't need to understand how it works, just how to use it:

In [ ]:
def minor(M, i, j):
    # Keep every row except row i...
    rows = [M[r] for r in range(M.rows) if r != i]
    # ...and in each of those rows, every entry except column j
    smaller = [[row[c] for c in range(M.cols) if c != j] for row in rows]
    return pan.Matrix(smaller)


# Example: delete row 0 and column 1 from A
minor(A, 0, 1)

Write `det3` using `minor` and **your own** `det2`. Watch the signs: $+,\ -,\ +$.

In [ ]:
def det3(M):
    # TODO: expand along the first row
    return ___


examples = {
    "Example 3.3": A,
    "Example 3.4": RowSwap(0, 2).apply(A),
    "Example 3.11": B,
}

for name, M in examples.items():
    try:
        mine = det3(M)
    except NameError:
        print("There's still a ___ to fill in inside det3!")
        break
    expected = pan.determinant(M)
    status = "PASS" if mine == expected else "NOT YET"
    print(f"{name}:  det3 = {mine},  panchi = {expected}  ->  {status}")

**Bonus.** Write a function `det(M)` that works for a square matrix of **any** size. Hint: a $1 \times 1$ matrix has determinant `M[0, 0]`; for bigger ones, expand along the first row and call `det` on each minor (a function is allowed to call itself!). The $(-1)^{j}$ sign can be written `(-1) ** j`. Test it on the $4 \times 4$ matrix from Example 3.7.

In [ ]:
def det(M):
    # TODO (bonus)
    pass


example_3_7 = pan.Matrix([
    [5,  4, 3, 2],
    [0,  1, 2, 0],
    [0, -1, 1, 0],
    [0,  1, 1, 3],
])

if det(example_3_7) is None:
    print("Bonus not attempted yet.")
else:
    print("det    =", det(example_3_7))
    print("panchi =", pan.determinant(example_3_7))

## Export to PDF

When you've finished everything above:

1. **Save** the notebook (**File → Save**, or Ctrl/Cmd + S). The PDF is made from the *saved* copy in your Drive.
2. Run the cell below.

Your PDF will appear in the main folder of your Google Drive (**My Drive**). The setup and export cells won't show up in the PDF.

In [ ]:
# Code and outputs are shown, In [ ]: prompts are hidden, "remove_cell" cells are left out,
# and "remove_input" cells only show their output
!jupyter nbconvert --to pdf "{NOTEBOOK}" --output-dir "{OUTPUT_FOLDER}" --TemplateExporter.exclude_input_prompt=True --TemplateExporter.exclude_output_prompt=True --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags=remove_cell --TagRemovePreprocessor.remove_input_tags=remove_input